# Offline gesture CNN training

Three runs: **Shallow CNN**, **Medium CNN**, **ResNet-18 (ImageNet transfer)**. Pick **best** by validation **accuracy** (tie-break: lower val **loss**). Writes **ONNX** + `label_map.json` to `artifacts/best/`; each full run stays under `artifacts/experiments/<name>/`.

**Data (HaGRID / ImageFolder)**: default root is `data/hagrid` with `train/<class>/` and `val/<class>/` (same layout as `data/README.md`). Official [HaGRID](https://github.com/hukenovs/hagrid) archives are per gesture; you still need a **train/val folder tree** (subset is fine). Toy random images are **opt-in** only (`USE_TOY_DATA = True` in Section 1) for pipeline smoke tests—not a substitute for HaGRID.

**How to run**: Open this notebook from the `offline-training` folder (so `Path.cwd()` is correct). Run cells top to bottom.

Docs: `docs/en/README.md` · `docs/zh-Hant/README.md`


## 0. Paths


In [ ]:
from __future__ import annotations

import json
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
ARTIFACTS = PROJECT_ROOT / "artifacts"
EXP_ROOT = ARTIFACTS / "experiments"
BEST_ROOT = ARTIFACTS / "best"

EXP_ROOT.mkdir(parents=True, exist_ok=True)
BEST_ROOT.mkdir(parents=True, exist_ok=True)

_mps = getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| mps:", _mps)


## 1. Hyperparameters


In [ ]:
# HaGRID (or any dataset): ImageFolder roots — see `data/README.md`.
# Point this at a folder that contains `train/<class>/` and `val/<class>/`.
DATA_DIR = PROJECT_ROOT / "data" / "hagrid"

# Professor requirement: train long enough for the schedule to anneal and val curves to stabilize.
# More epochs help with large/noisy data (HaGRID); shallow nets especially benefit from longer runs.
EPOCHS = 100

# Set True only to verify the notebook without real images (random RGB noise — not meaningful accuracy).
USE_TOY_DATA = False

BATCH_SIZE = 32
SEED = 42
IMAGE_SIZE = 224
DEVICE = None  # "cuda" | "mps" | "cpu" | None -> auto: cuda > mps > cpu

EXPERIMENTS = [
    {"experiment_name": "exp01_shallow_cnn", "architecture": "shallow", "lr": 3e-4},
    {"experiment_name": "exp02_medium_cnn", "architecture": "medium", "lr": 2e-4},
    {"experiment_name": "exp03_resnet18_transfer", "architecture": "resnet18", "lr": 1e-4},
]


## 2. Resolve `DATA_DIR` (HaGRID ImageFolder or optional toy)


In [ ]:
def make_toy_dataset(
    out: Path,
    *,
    n_classes: int = 5,
    per_class: int = 80,
    val_fraction: float = 0.2,
    img_size: int = 128,
    seed: int = 0,
) -> None:
    rng = np.random.default_rng(seed)
    out = out.resolve()
    if out.exists():
        shutil.rmtree(out)
    n_val = max(1, int(per_class * val_fraction))
    n_train = max(1, per_class - n_val)

    def write_random_png(path: Path) -> None:
        arr = rng.integers(0, 256, (img_size, img_size, 3), dtype=np.uint8)
        Image.fromarray(arr).save(path)

    for ci in range(n_classes):
        label = f"class_{ci:02d}"
        for i in range(n_train):
            d = out / "train" / label
            d.mkdir(parents=True, exist_ok=True)
            write_random_png(d / f"img_{i:04d}.png")
        for i in range(n_val):
            d = out / "val" / label
            d.mkdir(parents=True, exist_ok=True)
            write_random_png(d / f"img_{i:04d}.png")
    print(f"toy: {n_classes} classes, train {n_train}/class, val {n_val}/class -> {out}")


data_root = Path(DATA_DIR).resolve()
if USE_TOY_DATA:
    toy_path = PROJECT_ROOT / "data" / "toy"
    make_toy_dataset(toy_path, n_classes=5, per_class=60, img_size=128)
    DATA_DIR = toy_path
else:
    train_dir = data_root / "train"
    val_dir = data_root / "val"
    if not train_dir.is_dir() or not val_dir.is_dir():
        raise FileNotFoundError(
            "Expected ImageFolder layout for HaGRID (or any dataset):\n"
            f"  {data_root}/train/<class_name>/*.jpg\n"
            f"  {data_root}/val/<class_name>/*.jpg\n"
            "Official HaGRID zips are per-gesture; assign images into train/val (by subject split is best).\n"
            "See: data/README.md · https://github.com/hukenovs/hagrid\n"
            "To run without real data, set USE_TOY_DATA = True in the hyperparameters cell (smoke test only)."
        )

print("DATA_DIR =", Path(DATA_DIR).resolve())


## 3. Models and dataloaders


In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x).flatten(1))


class MediumCNN(nn.Module):
    def __init__(self, num_classes: int, dropout: float = 0.35):
        super().__init__()

        def block(cin: int, cout: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(block(3, 48), block(48, 96), block(96, 192), nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(192, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x).flatten(1))


def build_resnet18_transfer(num_classes: int) -> nn.Module:
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def build_model(architecture: str, num_classes: int) -> nn.Module:
    k = architecture.lower().strip()
    if k == "shallow":
        return ShallowCNN(num_classes)
    if k == "medium":
        return MediumCNN(num_classes)
    if k in ("resnet18", "resnet18_transfer"):
        return build_resnet18_transfer(num_classes)
    raise ValueError(architecture)


def build_loaders(data_root: Path, batch_size: int, image_size: int = 224):
    train_dir = data_root / "train"
    val_dir = data_root / "val"
    if not train_dir.is_dir() or not val_dir.is_dir():
        raise FileNotFoundError(f"Expected ImageFolder roots: {train_dir} and {val_dir}")

    train_tf = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.08, 0.08, 0.08, 0.02),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    val_tf = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    train_ds = datasets.ImageFolder(str(train_dir), transform=train_tf)
    val_ds = datasets.ImageFolder(str(val_dir), transform=val_tf)
    if train_ds.classes != val_ds.classes:
        raise ValueError("train and val class folders must match")

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader, train_ds.classes


## 4. Train, export ONNX, save under experiments


In [ ]:
@dataclass
class TrainConfig:
    data_dir: Path
    experiment_name: str
    architecture: str
    epochs: int
    batch_size: int
    lr: float
    seed: int
    image_size: int
    output_dir: Path
    device: str | None = None


def resolve_device(explicit: str | None) -> torch.device:
    if explicit:
        return torch.device(explicit)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate(model: nn.Module, loader, device: torch.device):
    model.eval()
    ce = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = ce(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def train_one(cfg: TrainConfig) -> dict[str, Any]:
    device = resolve_device(cfg.device)
    set_seed(cfg.seed)

    train_loader, val_loader, class_names = build_loaders(
        cfg.data_dir, batch_size=cfg.batch_size, image_size=cfg.image_size
    )
    num_classes = len(class_names)
    model = build_model(cfg.architecture, num_classes).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(cfg.epochs, 1))
    ce = nn.CrossEntropyLoss()

    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)

    best_key = (-1.0, -1e9)
    best_state = None

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        for x, y in tqdm(train_loader, desc=f"{cfg.experiment_name} e{epoch}/{cfg.epochs}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = ce(model(x), y)
            loss.backward()
            opt.step()
        sched.step()

        val_loss, val_acc = evaluate(model, val_loader, device)
        key = (val_acc, -val_loss)
        if key > best_key:
            best_key = key
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"  [{cfg.experiment_name}] ep{epoch}: val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    val_loss, val_acc = evaluate(model, val_loader, device)

    torch.save({"model": model.state_dict(), "classes": class_names}, out / "checkpoint.pt")
    metrics = {
        "experiment_name": cfg.experiment_name,
        "architecture": cfg.architecture,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "num_classes": num_classes,
        "class_names": class_names,
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "seed": cfg.seed,
        "image_size": cfg.image_size,
    }
    (out / "metrics.json").write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    model.eval()
    # ONNX: export on CPU for broader runtime compatibility (esp. MPS).
    onnx_model = model.cpu()
    dummy = torch.zeros(1, 3, cfg.image_size, cfg.image_size)
    torch.onnx.export(
        onnx_model,
        dummy,
        str(out / "gesture.onnx"),
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17,
    )

    label_map = {str(i): name for i, name in enumerate(class_names)}
    (out / "label_map.json").write_text(
        json.dumps(label_map, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    meta = {
        **metrics,
        "onnx_opset": 17,
        "input_layout": "NCHW",
        "input_channels": 3,
        "input_height": cfg.image_size,
        "input_width": cfg.image_size,
        "export": "onnx",
    }
    (out / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return metrics


def promote_to_best(src_dir: Path, best_dir: Path) -> None:
    best_dir.mkdir(parents=True, exist_ok=True)
    for name in ("gesture.onnx", "label_map.json", "meta.json", "metrics.json"):
        p = src_dir / name
        if p.is_file():
            shutil.copy2(p, best_dir / name)


## 5. Run all experiments and promote best to `artifacts/best/`


In [ ]:
leaderboard: list[dict] = []
data_dir = DATA_DIR.resolve()

for spec in EXPERIMENTS:
    name = spec["experiment_name"]
    out_dir = EXP_ROOT / name
    cfg = TrainConfig(
        data_dir=data_dir,
        experiment_name=name,
        architecture=spec["architecture"],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=float(spec["lr"]),
        seed=SEED,
        image_size=IMAGE_SIZE,
        output_dir=out_dir,
        device=DEVICE,
    )
    leaderboard.append(train_one(cfg))

(EXP_ROOT / "leaderboard.json").write_text(
    json.dumps(leaderboard, indent=2, ensure_ascii=False), encoding="utf-8"
)

best = max(leaderboard, key=lambda m: (m["val_accuracy"], -m["val_loss"]))
promote_to_best(EXP_ROOT / best["experiment_name"], BEST_ROOT)

summary = {
    "primary_metric": "val_accuracy",
    "tie_break": "max val_accuracy, then lower val_loss",
    "winner": best["experiment_name"],
    "val_accuracy": best["val_accuracy"],
    "val_loss": best["val_loss"],
    "artifacts_best": str(BEST_ROOT),
}
(BEST_ROOT / "selection.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n=== Leaderboard ===")
for m in sorted(leaderboard, key=lambda x: (-x["val_accuracy"], x["val_loss"])):
    print(f"  {m['experiment_name']}: acc={m['val_accuracy']:.4f} loss={m['val_loss']:.4f}")
print(f"\nBest copied to {BEST_ROOT}")
